# CS5494 Final Project — Restaurant Review Rating

本 Notebook 按照实验流程逐步完成三条评分路线：

| Task | 方法 | 说明 |
|------|------|------|
| **A** | Zero-shot | 直接用基础模型推理 |
| **B.1.1** | N-shot ICL | 给模型提供少量示例后推理 |
| **B.1.2** | LoRA 微调 | 指令微调后推理 |

**运行前请确认**：
- 已执行 `pip install -r requirements.txt`
- `data/` 目录下包含 `review_train.csv`、`review_test.csv`、`test_answer.csv`
- （可选）如需从 ModelScope 下载模型，请设置 `USE_MODELSCOPE = True`

---
## 0. 全局配置

所有超参数、路径、开关统一在此 cell 设置，后续 cell 均使用这里的变量。

In [6]:
import sys, os
# notebook 在 notebooks/ 子目录，把项目根目录加入 Python 路径
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

MODEL_NAME      = "Qwen/Qwen2.5-0.5B-Instruct"
# MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME      = "Qwen/Qwen2.5-3B-Instruct"
USE_MODELSCOPE  = True                            # True = ModelScope 镜像下载
MODEL_CACHE_DIR = "../models"

DATA_DIR        = "../data"
SEED            = 5494

DEBUG_TRAIN_SIZE = None
DEBUG_TEST_SIZE  = None

N_SHOT          = 4        # few-shot 示例数
N_PER_RATING    = 10       # 每个评分从训练集中收集多少个示例

LORA_CHECKPOINT_DIR = "../qwen_lora_checkpoints"
LORA_FINAL_DIR      = "../qwen_lora_final"

MAX_NEW_TOKENS  = 10       # 推理时最多生成 token 数

model_short_name  = MODEL_NAME.split('/')[-1]
OUTPUT_DIR      = f"../outputs/temp/{model_short_name}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"配置完成 ✓ (结果将保存在 {OUTPUT_DIR})")


配置完成 ✓ (结果将保存在 ../outputs/temp/Qwen2.5-0.5B-Instruct)


---
## 1. 数据加载

`load_project_data` 会自动读取三份 CSV 并将文本评分（如 `"4 star"`）转换为数值列 `rating_numeric`（1-5）。

In [7]:
from dataset import load_project_data

train_df, test_df, answer_df = load_project_data(DATA_DIR)

print(f"训练集: {len(train_df)} 条")
print(f"测试集: {len(test_df)} 条")
print(f"答案集: {len(answer_df)} 条")
train_df.head(3)

训练集: 9067 条
测试集: 1007 条
答案集: 1007 条


,Review_id,Author,Title,Review,Rating,Dates,Restaurant,Location,Category,rating_numeric
0,4091810389,Choi Yee C,no table,"nice place to visit everytime go melaka, not s...",4 star,Reviewed 14 June 2016,Jonker 86 QQ Ice,Melaka,Italian,4
1,8163808786,Renganathan S,Best food in town,I tried linguine bolognaise sauce... It's very...,5 star,Reviewed 13 September 2013,Gianni's Trattoria,JB,Italian,5
2,6466234202,"vidyamenonKuala Lumpur, Malaysia",Was in the area...,Was around the area and stopped by to have a d...,4 star,Reviewed 14 April 2018,One Serambi Cafe,Shah Alam,Mediterranean,4


### 1.1 标签分布

查看训练集各评分类别的样本数，确认数据是否均衡。

In [8]:
print("训练集评分分布:")
print(train_df["rating_numeric"].value_counts().sort_index())
print()
print("答案集评分分布:")
print(answer_df["rating_numeric"].value_counts().sort_index())

训练集评分分布:
rating_numeric
1    1817
2    1951
3    1739
4    1785
5    1775
Name: count, dtype: int64

答案集评分分布:
rating_numeric
1    212
2    223
3    213
4    169
5    190
Name: count, dtype: int64


---
## 2. 划分训练集 / 验证集

使用分层切分（stratify）保证各评分类别在训练集和验证集中比例一致。

- `test_size=0.2`：20% 作为验证集
- `debug_sample_size`：设置后会对切分结果进行随机截断，用于快速调试

In [9]:
from dataset import split_train_val

train_split, val_split = split_train_val(
    train_df,
    seed=SEED,
    test_size=0.2,
    debug_sample_size=DEBUG_TRAIN_SIZE,
)

print(f"训练子集: {len(train_split)} 条")
print(f"验证子集: {len(val_split)} 条")

训练子集: 7253 条
验证子集: 1814 条


### 调试用：截断测试集

推理很慢时，可以先在小样本上验证流程。

In [10]:
if DEBUG_TEST_SIZE is not None:
    test_df  = test_df.sample(n=min(DEBUG_TEST_SIZE, len(test_df)), random_state=SEED).reset_index(drop=True)
    answer_df = answer_df[answer_df["Review_id"].isin(test_df["Review_id"])].reset_index(drop=True)
    print(f"调试模式：测试集截断为 {len(test_df)} 条")
else:
    print(f"全量测试集：{len(test_df)} 条")

全量测试集：1007 条


---
## 3. 加载基础模型

`load_tokenizer_and_model` 会自动检测设备（CUDA / NPU / MPS / CPU）并选择合适的 dtype。

返回值：
- `tokenizer`：分词器
- `base_model`：未微调的基础模型
- `model_path`：实际加载路径
- `device`：`DeviceInfo` 对象，含设备类型和显存信息

In [11]:
from model import load_tokenizer_and_model

tokenizer, base_model, model_path, device = load_tokenizer_and_model(
    model_name=MODEL_NAME,
    use_modelscope=USE_MODELSCOPE,
    cache_dir=MODEL_CACHE_DIR,
)

print(f"设备类型 : {device.device_type}")
print(f"设备名称 : {device.device_name}")
if device.total_memory_gb:
    print(f"显存容量 : {device.total_memory_gb:.1f} GB")
print(f"模型路径 : {model_path}")

d:\anaconda3\condaData\envs_dirs\qwen_lora\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


本地缓存命中，跳过下载: ..\models\Qwen\Qwen2.5-0.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2059.10it/s]


设备类型 : cuda
设备名称 : NVIDIA GeForce RTX 5060 Laptop GPU
显存容量 : 8.5 GB
模型路径 : ..\models\Qwen\Qwen2.5-0.5B-Instruct


---
## 4. Task A — Zero-shot 推理

直接使用基础模型，不提供任何示例。

**Prompt 格式**：描述评分标准 + 评论文本，要求模型输出单个数字（1-5）。

返回的 `zero_out` 是一个字典：
- `predictions_df`：含 `Review_id` 和 `Predicted_Rating` 的 DataFrame
- `debug_records`：每条样本的原始输出和解析详情
- `inference_seconds`：总推理耗时

In [12]:
import time
import pandas as pd
from model import build_zero_shot_prompt, generate_response, extract_rating_from_output

sample_row = test_df.iloc[6]
print(f'Review_id : {sample_row["Review_id"]}')
print(f'Title     : {sample_row.get("Title")}')
print(f'Review    : {str(sample_row["Review"])}')
true_answer = answer_df[answer_df['Review_id'] == sample_row['Review_id']]['rating_numeric'].values[0]
print(f'True_ans  : {true_answer}')


Review_id : 5123417064
Title     : Excellent except Terrible Staff
Review    : It was unplanned visit to this restaurant but we I and my friends was satisfied with the foods and drinks they served. But, one staff their who is thin and tall and if im not mistaken the name is Jayson he shows his rudeness to us when my lady friend calling his attention and we are pretty sure that with his awareness he just ignore and make a face to my friend. Then, when he passes in our table I notice he walked like a woman though his not by then I learned why he ignored my lady friend. We will try to comeback we hope we dont encounter him anymore.
True_ans  : 2


In [13]:
# 第 1 步：构建 prompt
sample_title = '' if pd.isna(sample_row.get('Title')) else str(sample_row.get('Title'))
sample_review = str(sample_row['Review'])
sample_prompt = build_zero_shot_prompt(title=sample_title, review=sample_review)
print(f'Prompt: \n'+sample_prompt)


Prompt: 
Given the following restaurant review, please rate it from 1 to 5 stars, where:
- 1 star: Very poor experience
- 2 stars: Poor experience
- 3 stars: Average experience
- 4 stars: Good experience
- 5 stars: Excellent experience

Title: Excellent except Terrible Staff
Review: It was unplanned visit to this restaurant but we I and my friends was satisfied with the foods and drinks they served. But, one staff their who is thin and tall and if im not mistaken the name is Jayson he shows his rudeness to us when my lady friend calling his attention and we are pretty sure that with his awareness he just ignore and make a face to my friend. Then, when he passes in our table I notice he walked like a woman though his not by then I learned why he ignored my lady friend. We will try to comeback we hope we dont encounter him anymore.

Please provide only a single number (1, 2, 3, 4, or 5) as your rating.
Return format: `Rating: <digit>` (no explanation).
Rating: 


In [14]:
# 第 2 步：调用模型生成
sample_raw_output = generate_response(
    model=base_model,
    tokenizer=tokenizer,
    prompt=sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'模型原始输出: '+ repr(sample_raw_output))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


模型原始输出: 'Rating: 2'


In [15]:
# 第 3 步：解析评分
sample_predicted = extract_rating_from_output(sample_raw_output)
print(f'解析结果: {sample_predicted}')

解析结果: 2


In [16]:
ids = []
predictions = []
debug_records = []

start_time = time.time()

for step, (_, row) in enumerate(test_df.iterrows()):
    review_id = row['Review_id']
    title = '' if pd.isna(row.get('Title')) else str(row.get('Title'))
    review = str(row['Review'])

    # 第 1 步：构建 prompt
    prompt = build_zero_shot_prompt(title=title, review=review)

    # 第 2 步：调用模型生成
    raw_output = generate_response(
        model=base_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 第 3 步：从模型输出中解析评分（1-5），解析失败默认为 3
    predicted_rating = extract_rating_from_output(raw_output)

    # 收集结果
    ids.append(review_id)
    predictions.append(predicted_rating)
    debug_records.append({
        'sample_idx': step,
        'review_id': review_id,
        'prompt': prompt,
        'raw_output': raw_output,
        'predicted_rating': predicted_rating,
    })

    # 每 10 条打印一次进度
    if (step + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f'已处理 {step + 1}/{len(test_df)} 条，耗时 {elapsed:.1f}s')

total_time = time.time() - start_time

# 汇总结果
zero_predictions_df = pd.DataFrame({'Review_id': ids, 'Predicted_Rating': predictions})

zero_out = {
    'predictions_df': zero_predictions_df,
    'debug_records': debug_records,
    'inference_seconds': total_time,
    'sample_count': len(predictions),
}

print(f'\n推理完成：{zero_out["sample_count"]} 条')
print(f'总耗时  ：{zero_out["inference_seconds"]:.1f}s')
print(f'平均耗时：{zero_out["inference_seconds"] / zero_out["sample_count"]:.2f}s/样本')
zero_out['predictions_df'].head()

已处理 10/1007 条，耗时 1.2s
已处理 20/1007 条，耗时 2.4s
已处理 30/1007 条，耗时 3.5s
已处理 40/1007 条，耗时 4.7s
已处理 50/1007 条，耗时 5.8s
已处理 60/1007 条，耗时 7.0s
已处理 70/1007 条，耗时 8.1s
已处理 80/1007 条，耗时 9.3s
已处理 90/1007 条，耗时 10.5s
已处理 100/1007 条，耗时 11.7s
已处理 110/1007 条，耗时 12.9s
已处理 120/1007 条，耗时 14.1s
已处理 130/1007 条，耗时 15.2s
已处理 140/1007 条，耗时 16.4s
已处理 150/1007 条，耗时 17.6s
已处理 160/1007 条，耗时 18.9s
已处理 170/1007 条，耗时 20.0s
已处理 180/1007 条，耗时 21.2s
已处理 190/1007 条，耗时 22.3s
已处理 200/1007 条，耗时 23.4s
已处理 210/1007 条，耗时 24.6s
已处理 220/1007 条，耗时 25.7s
已处理 230/1007 条，耗时 26.8s
已处理 240/1007 条，耗时 28.0s
已处理 250/1007 条，耗时 29.1s
已处理 260/1007 条，耗时 30.3s
已处理 270/1007 条，耗时 31.4s
已处理 280/1007 条，耗时 32.6s
已处理 290/1007 条，耗时 33.8s
已处理 300/1007 条，耗时 35.0s
已处理 310/1007 条，耗时 36.2s
已处理 320/1007 条，耗时 37.3s
已处理 330/1007 条，耗时 38.5s
已处理 340/1007 条，耗时 39.6s
已处理 350/1007 条，耗时 40.8s
已处理 360/1007 条，耗时 42.0s
已处理 370/1007 条，耗时 43.1s
已处理 380/1007 条，耗时 44.3s
已处理 390/1007 条，耗时 45.5s
已处理 400/1007 条，耗时 46.6s
已处理 410/1007 条，耗时 47.8s
已处理 420/1007 条，耗时 48.9s
已处理 430/1

,Review_id,Predicted_Rating
0,6204917469,1
1,7226897287,1
2,1264902203,2
3,3692481359,2
4,7677228495,5


### 4.1 评估 Zero-shot 结果

In [17]:
from eval import evaluate_predictions

zero_metric = evaluate_predictions(zero_out["predictions_df"], answer_df)

print(f"Accuracy    : {zero_metric['accuracy']:.4f}")
print(f"Macro-F1    : {zero_metric['macro_f1']:.4f}")
print(f"Weighted-F1 : {zero_metric['weighted_f1']:.4f}")
print()
print(zero_metric["classification_report"])

Parse fallback rate: N/A (no parser tracking metadata)
Accuracy    : 0.4816
Macro-F1    : 0.4537
Weighted-F1 : 0.4485

              precision    recall  f1-score   support

      1 star       0.47      0.96      0.63       212
      2 star       0.30      0.24      0.27       223
      3 star       0.43      0.20      0.27       213
      4 star       0.47      0.34      0.39       169
      5 star       0.74      0.67      0.71       190

    accuracy                           0.48      1007
   macro avg       0.48      0.48      0.45      1007
weighted avg       0.47      0.48      0.45      1007



### 4.2 混淆矩阵

In [18]:
import pandas as pd

cm_zero = pd.DataFrame(
    zero_metric["confusion_matrix"],
    index=[f"True {i}★" for i in range(1, 6)],
    columns=[f"Pred {i}★" for i in range(1, 6)],
)
print("Zero-shot 混淆矩阵:")
cm_zero

Zero-shot 混淆矩阵:


,Pred 1★,Pred 2★,Pred 3★,Pred 4★,Pred 5★
True 1★,204,8,0,0,0
True 2★,168,54,1,0,0
True 3★,56,98,42,15,2
True 4★,4,19,46,57,43
True 5★,1,4,9,48,128


### 4.3 保存预测结果

In [19]:
zero_out["predictions_df"].to_csv(f"{OUTPUT_DIR}/zero_shot_predictions.csv", index=False)
print("已保存 zero_shot_predictions.csv")

已保存 zero_shot_predictions.csv


---
## 5. Task B.1.1 — N-shot In-Context Learning

在同一基础模型上，在 prompt 中插入 N 个带标注的示例（few-shot examples），引导模型仿照示例格式输出评分。

**示例库构建**：`build_example_library` 从训练集中每个评分等级各采样 `n_per_rating` 条，共最多 5×N 条。

推理时每条测试样本会从示例库中随机选 `n_shot` 条插入 prompt，使用确定性 hash seed 保证可复现。

In [15]:
import pandas as pd
import hashlib
from dataset import build_example_library
from model import build_nshot_prompt, generate_response, extract_rating_from_output

# 构建示例库
example_lib = build_example_library(train_split, n_per_rating=N_PER_RATING, seed=SEED)
print(f"示例库大小：{len(example_lib)} 条（每评分最多 {N_PER_RATING} 条）")

示例库大小：50 条（每评分最多 10 条）


In [16]:
# 取测试集第 1 行
sample_row = test_df.iloc[6]
review_id = sample_row["Review_id"]
print(f'Review_id : {review_id}')
print(f'Title     : {sample_row.get("Title")}')
print(f'Review    : {str(sample_row["Review"])}')
true_answer = answer_df[answer_df['Review_id'] == sample_row['Review_id']]['rating_numeric'].values[0]
print(f'True_ans  : {true_answer}')

Review_id : 5123417064
Title     : Excellent except Terrible Staff
Review    : It was unplanned visit to this restaurant but we I and my friends was satisfied with the foods and drinks they served. But, one staff their who is thin and tall and if im not mistaken the name is Jayson he shows his rudeness to us when my lady friend calling his attention and we are pretty sure that with his awareness he just ignore and make a face to my friend. Then, when he passes in our table I notice he walked like a woman though his not by then I learned why he ignored my lady friend. We will try to comeback we hope we dont encounter him anymore.
True_ans  : 2


In [17]:
# 第 1 步：哈希计算种子 & 构建 prompt
review_id_text = str(review_id)
stable_hash = int(hashlib.md5(review_id_text.encode("utf-8")).hexdigest()[:8], 16)
row_seed = SEED + (stable_hash % 1_000_000)

sample_title = '' if pd.isna(sample_row.get('Title')) else str(sample_row.get('Title'))
sample_review = str(sample_row['Review'])
sample_prompt = build_nshot_prompt(
    title=sample_title, 
    review=sample_review, 
    examples=example_lib,
    n=N_SHOT,
    seed=row_seed
)
print(f'动态种子: {row_seed}')
print(f'N-shot Prompt: \n' + sample_prompt)

动态种子: 584439
N-shot Prompt: 
Given the following restaurant review, please rate it from 1 to 5 stars, where:
- 1 star: Very poor experience
- 2 stars: Poor experience
- 3 stars: Average experience
- 4 stars: Good experience
- 5 stars: Excellent experience

Here are some examples as below:
### Example 1
Title: Poor Choice of Restaurant
Review: We went last weekend on a Saturday, around 7pm. We came earlier so as to avoid the crowd. We just wanted to have a nice, quiet dinner with a lovely view. Unfortunately, one of our main dish did not turn out right. It is probably the worst Steak I've ever tried. It's tough and chewy that we had to send it back to the kitchen and ask the chef to rectify it. The chef just had to cook it again and gue...
Rating: 2

### Example 2
Title: Coming Home
Review: This is THE place for Sarawak laksa when I come back to my home town. Sarawakians have their favourite laksa haunts and this is mine because of my childhood memories of coming here on Sundays after m

In [18]:
# 第 2 步：调用模型生成
sample_raw_output = generate_response(
    model=base_model,
    tokenizer=tokenizer,
    prompt=sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'模型原始输出: '+repr(sample_raw_output))

模型原始输出: 'Rating: 5'


In [19]:
# 第 3 步：解析评分
sample_predicted = extract_rating_from_output(sample_raw_output)
print(f'解析结果: {sample_predicted}')

解析结果: 5


In [20]:
ids = []
predictions = []
debug_records = []

start_time = time.time()

for step, (_, row) in enumerate(test_df.iterrows()):
    review_id = row['Review_id']
    title = '' if pd.isna(row.get('Title')) else str(row.get('Title'))
    review = str(row['Review'])

    review_id_text = str(review_id)
    stable_hash = int(hashlib.md5(review_id_text.encode("utf-8")).hexdigest()[:8], 16)
    row_seed = SEED + (stable_hash % 1_000_000)

    # 第 1 步：构建 prompt
    prompt = build_nshot_prompt(
        title=title, 
        review=review, 
        examples=example_lib,
        n=N_SHOT,
        seed=row_seed
    )

    # 第 2 步：调用模型生成
    raw_output = generate_response(
        model=base_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 第 3 步：从模型输出中解析评分（1-5），解析失败默认为 3
    predicted_rating = extract_rating_from_output(raw_output)

    # 收集结果
    ids.append(review_id)
    predictions.append(predicted_rating)
    debug_records.append({
        'sample_idx': step,
        'review_id': review_id,
        'prompt': prompt,
        'raw_output': raw_output,
        'predicted_rating': predicted_rating,
    })

    # 每 10 条打印一次进度
    if (step + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f'已处理 {step + 1}/{len(test_df)} 条，耗时 {elapsed:.1f}s')

total_time = time.time() - start_time

# 汇总结果
nshot_predictions_df = pd.DataFrame({'Review_id': ids, 'Predicted_Rating': predictions})

nshot_out = {
    'predictions_df': nshot_predictions_df,
    'debug_records': debug_records,
    'inference_seconds': total_time,
    'sample_count': len(predictions),
}

print(f'\n推理完成：{nshot_out["sample_count"]} 条')
print(f'总耗时  ：{nshot_out["inference_seconds"]:.1f}s')
print(f'平均耗时：{nshot_out["inference_seconds"] / nshot_out["sample_count"]:.2f}s/样本')


已处理 10/1007 条，耗时 1.3s
已处理 20/1007 条，耗时 2.7s
已处理 30/1007 条，耗时 4.1s
已处理 40/1007 条，耗时 5.4s
已处理 50/1007 条，耗时 6.7s
已处理 60/1007 条，耗时 8.1s
已处理 70/1007 条，耗时 9.4s
已处理 80/1007 条，耗时 10.8s
已处理 90/1007 条，耗时 12.1s
已处理 100/1007 条，耗时 13.5s
已处理 110/1007 条，耗时 14.8s
已处理 120/1007 条，耗时 16.2s
已处理 130/1007 条，耗时 17.6s
已处理 140/1007 条，耗时 18.9s
已处理 150/1007 条，耗时 20.3s
已处理 160/1007 条，耗时 21.8s
已处理 170/1007 条，耗时 23.1s
已处理 180/1007 条，耗时 24.4s
已处理 190/1007 条，耗时 25.6s
已处理 200/1007 条，耗时 26.9s
已处理 210/1007 条，耗时 28.2s
已处理 220/1007 条，耗时 29.6s
已处理 230/1007 条，耗时 30.9s
已处理 240/1007 条，耗时 32.3s
已处理 250/1007 条，耗时 33.8s
已处理 260/1007 条，耗时 35.2s
已处理 270/1007 条，耗时 36.5s
已处理 280/1007 条，耗时 37.7s
已处理 290/1007 条，耗时 39.0s
已处理 300/1007 条，耗时 40.4s
已处理 310/1007 条，耗时 41.7s
已处理 320/1007 条，耗时 43.0s
已处理 330/1007 条，耗时 44.4s
已处理 340/1007 条，耗时 45.8s
已处理 350/1007 条，耗时 47.1s
已处理 360/1007 条，耗时 48.4s
已处理 370/1007 条，耗时 49.8s
已处理 380/1007 条，耗时 51.2s
已处理 390/1007 条，耗时 52.5s
已处理 400/1007 条，耗时 53.9s
已处理 410/1007 条，耗时 55.3s
已处理 420/1007 条，耗时 56.6s
已处理 430/

### 5.1 评估 N-shot 结果

In [21]:
nshot_metric = evaluate_predictions(nshot_out["predictions_df"], answer_df)

print(f"Accuracy    : {nshot_metric['accuracy']:.4f}")
print(f"Macro-F1    : {nshot_metric['macro_f1']:.4f}")
print(f"Weighted-F1 : {nshot_metric['weighted_f1']:.4f}")
print()
print(nshot_metric["classification_report"])

Accuracy    : 0.5065
Macro-F1    : 0.4390
Weighted-F1 : 0.4446

              precision    recall  f1-score   support

      1 star       0.52      0.77      0.62       212
      2 star       0.50      0.14      0.22       223
      3 star       0.52      0.63      0.57       213
      4 star       0.50      0.09      0.16       169
      5 star       0.49      0.87      0.62       190

    accuracy                           0.51      1007
   macro avg       0.51      0.50      0.44      1007
weighted avg       0.51      0.51      0.44      1007



### 5.2 混淆矩阵

In [22]:
cm_nshot = pd.DataFrame(
    nshot_metric["confusion_matrix"],
    index=[f"True {i}★" for i in range(1, 6)],
    columns=[f"Pred {i}★" for i in range(1, 6)],
)
print(f"{N_SHOT}-shot 混淆矩阵:")
cm_nshot

4-shot 混淆矩阵:


,Pred 1★,Pred 2★,Pred 3★,Pred 4★,Pred 5★
True 1★,163,10,8,2,29
True 2★,119,32,49,4,19
True 3★,27,21,134,4,27
True 4★,1,1,53,16,98
True 5★,5,0,14,6,165


### 5.3 保存预测结果

In [23]:
nshot_out["predictions_df"].to_csv(f"{OUTPUT_DIR}/nshot_predictions.csv", index=False)
print(f"已保存 nshot_predictions.csv")

已保存 nshot_predictions.csv


---
## 6. Task B.1.2 — LoRA 微调

对基础模型进行指令微调（SFT），使用 PEFT 的 LoRA 方法只训练少量参数。

**训练数据格式**（`prepare_instruction_data` 的输出）：
```
{ "instruction": "...", "input": "Title: ...\nReview: ...", "output": "4" }
```

**关键超参数**（可在 `LoraTrainingConfig` 中修改）：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `lora_rank` | 4 | LoRA 低秩矩阵的秩 |
| `lora_alpha` | 8 | LoRA 缩放系数 |
| `num_train_epochs` | 3 | 训练轮数 |
| `train_batch_size` | 8 | 每个设备 batch 大小 |
| `learning_rate` | 5e-5 | 学习率 |

In [24]:
from dataset import prepare_instruction_data
from train import LoraTrainingConfig, train_lora_model

# 构建指令样本
train_records = prepare_instruction_data(train_split)
val_records   = prepare_instruction_data(val_split)
print(f"训练指令样本: {len(train_records)} 条")
print(f"验证指令样本: {len(val_records)} 条")

# 示例：打印第一条样本
print("\n第一条训练样本:")
for k, v in train_records[0].items():
    print(f"{k}: {v[:80]}{'...' if len(v) > 80 else ''}")

训练指令样本: 7253 条
验证指令样本: 1814 条

第一条训练样本:
instruction: Given the following restaurant review, please rate it from 1 to 5 stars, where:
...
input: Title: A proper Indian
Review: It's not memorable or outstanding, but it is a go...
output: 4


### 6.1 配置 & 启动训练

> ⚠️ **注意**：训练会覆盖 `base_model` 的 adapter 状态。若需重跑 Task A/B.1.1，请重新执行 Section 3 重新加载基础模型。

In [25]:
cfg = LoraTrainingConfig(
    output_dir=LORA_CHECKPOINT_DIR,
    final_dir=LORA_FINAL_DIR,
    # 调试时可减少训练样本和 epoch
    # max_train_samples=100,
    # num_train_epochs=1,
)

train_res = train_lora_model(
    base_model=base_model,
    tokenizer=tokenizer,
    train_records=train_records,
    val_records=val_records,
    config=cfg,
)

print(f"训练完成 ✓")
print(f"总耗时      : {train_res['train_seconds']:.1f}s")
print(f"训练样本数  : {train_res['train_samples']}")
print(f"验证样本数  : {train_res['val_samples']}")
print(f"模型保存路径: {train_res['output_dir']}")

Epoch,Training Loss,Validation Loss
1,1.655470,1.565817
2,1.554926,1.557953
3,1.549345,1.555783


训练完成 ✓
总耗时      : 16765.5s
训练样本数  : 7253
验证样本数  : 1814
模型保存路径: ../qwen_lora_final


---
## 7. Task B.1.2 — LoRA 推理

将 LoRA adapter 合并（merge）到基础模型权重中，再进行推理。

**Merge 的好处**：推理速度与原始模型相同，无 adapter 额外开销。

> 若已有训练好的 adapter，可以直接修改 `lora_dir` 跳过 Section 6 直接执行本节。

In [26]:
from model import load_merged_lora_model

lora_dir = train_res["output_dir"]

tok_lora, merged_model, lora_device = load_merged_lora_model(
    base_model_name=MODEL_NAME,
    lora_dir=lora_dir,
    use_modelscope=USE_MODELSCOPE,
    cache_dir=MODEL_CACHE_DIR,
)

print(f"LoRA merge 完成 ✓  设备: {lora_device.device_type}")

本地缓存命中，跳过下载: ..\models\Qwen\Qwen2.5-0.5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1871.96it/s]


LoRA merge 完成 ✓  设备: cuda


## LoRA微调后推理

In [27]:
import pandas as pd
from model import build_finetuned_prompt, generate_response, extract_rating_from_output

# 取测试集
sample_row = test_df.iloc[6]
review_id = sample_row["Review_id"]
print(f'Review_id : {review_id}')
print(f'Title     : {sample_row.get("Title")}')
print(f'Review    : {str(sample_row["Review"])[:200]}...')
true_answer = answer_df[answer_df['Review_id'] == sample_row['Review_id']]['rating_numeric'].values[0]
print(f'True_ans  : {true_answer}')

Review_id : 5123417064
Title     : Excellent except Terrible Staff
Review    : It was unplanned visit to this restaurant but we I and my friends was satisfied with the foods and drinks they served. But, one staff their who is thin and tall and if im not mistaken the name is Jays...
True_ans  : 2


In [28]:
# 第 1 步：构建微调模型专用的 Prompt
sample_title = '' if pd.isna(sample_row.get('Title')) else str(sample_row.get('Title'))
sample_review = str(sample_row['Review'])
sample_prompt = build_finetuned_prompt(title=sample_title, review=sample_review)
print(f'Zero-shot Prompt:\n' + sample_prompt)

Zero-shot Prompt:
Given the following restaurant review, please rate it from 1 to 5 stars, where:
- 1 star: Very poor experience
- 2 stars: Poor experience
- 3 stars: Average experience
- 4 stars: Good experience
- 5 stars: Excellent experience

Title: Excellent except Terrible Staff
Review: It was unplanned visit to this restaurant but we I and my friends was satisfied with the foods and drinks they served. But, one staff their who is thin and tall and if im not mistaken the name is Jayson he shows his rudeness to us when my lady friend calling his attention and we are pretty sure that with his awareness he just ignore and make a face to my friend. Then, when he passes in our table I notice he walked like a woman though his not by then I learned why he ignored my lady friend. We will try to comeback we hope we dont encounter him anymore.

Please provide only a single number (1, 2, 3, 4, or 5) as your rating.
Return format: `Rating: <digit>` (no explanation).
Rating: 


In [ ]:
# 调用微调后的模型生成（挂载 LoRA 权重）
sample_raw_output = generate_response(
    model=merged_model,
    tokenizer=tok_lora,
    prompt=sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'模型原始输出: ' + repr(sample_raw_output))

模型原始输出: '2'


In [30]:
# 第 3 步：解析评级数字
sample_predicted = extract_rating_from_output(sample_raw_output)
print(f'Rating:  {sample_predicted}')

Rating:  2


In [31]:
import time
import pandas as pd
from model import build_finetuned_prompt, generate_response, extract_rating_from_output

ids = []
predictions = []
debug_records = []

start_time = time.time()

for step, (_, row) in enumerate(test_df.iterrows()):
    review_id = row['Review_id']
    title = '' if pd.isna(row.get('Title')) else str(row.get('Title'))
    review = str(row['Review'])

    # 第 1 步：构建 Prompt (Zero-shot)
    prompt = build_finetuned_prompt(title=title, review=review)

    # 第 2 步：调用模型生成
    raw_output = generate_response(
        model=merged_model,
        tokenizer=tok_lora,       
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 第 3 步：解析评分
    predicted_rating = extract_rating_from_output(raw_output)

    # 收集结果
    ids.append(review_id)
    predictions.append(predicted_rating)
    debug_records.append({
        'sample_idx': step,
        'review_id': review_id,
        'prompt': prompt,
        'raw_output': raw_output,
        'predicted_rating': predicted_rating,
    })

    # 每 10 条打印一次进度
    if (step + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f'已处理 {step + 1}/{len(test_df)} 条，耗时 {elapsed:.1f}s')

total_time = time.time() - start_time

# 汇总结果打包
lora_predictions_df = pd.DataFrame({'Review_id': ids, 'Predicted_Rating': predictions})
lora_out = {
    'predictions_df': lora_predictions_df,
    'debug_records': debug_records,
    'inference_seconds': total_time,
    'sample_count': len(predictions),
}

print(f'\n推理完成：{lora_out["sample_count"]} 条')
print(f'总耗时  ：{lora_out["inference_seconds"]:.1f}s')
print(f'平均耗时：{lora_out["inference_seconds"] / lora_out["sample_count"]:.2f}s/样本')

已处理 10/1007 条，耗时 1.1s
已处理 20/1007 条，耗时 2.0s
已处理 30/1007 条，耗时 7.8s
已处理 40/1007 条，耗时 8.8s
已处理 50/1007 条，耗时 9.9s
已处理 60/1007 条，耗时 10.8s
已处理 70/1007 条，耗时 11.7s
已处理 80/1007 条，耗时 18.2s
已处理 90/1007 条，耗时 19.2s
已处理 100/1007 条，耗时 20.1s
已处理 110/1007 条，耗时 21.0s
已处理 120/1007 条，耗时 22.2s
已处理 130/1007 条，耗时 23.1s
已处理 140/1007 条，耗时 24.1s
已处理 150/1007 条，耗时 25.3s
已处理 160/1007 条，耗时 46.7s
已处理 170/1007 条，耗时 47.5s
已处理 180/1007 条，耗时 48.4s
已处理 190/1007 条，耗时 49.3s
已处理 200/1007 条，耗时 50.3s
已处理 210/1007 条，耗时 51.2s
已处理 220/1007 条，耗时 52.3s
已处理 230/1007 条，耗时 53.3s
已处理 240/1007 条，耗时 54.2s
已处理 250/1007 条，耗时 55.2s
已处理 260/1007 条，耗时 56.2s
已处理 270/1007 条，耗时 57.1s
已处理 280/1007 条，耗时 58.0s
已处理 290/1007 条，耗时 58.9s
已处理 300/1007 条，耗时 59.9s
已处理 310/1007 条，耗时 61.0s
已处理 320/1007 条，耗时 62.0s
已处理 330/1007 条，耗时 62.9s
已处理 340/1007 条，耗时 63.9s
已处理 350/1007 条，耗时 64.8s
已处理 360/1007 条，耗时 72.3s
已处理 370/1007 条，耗时 73.5s
已处理 380/1007 条，耗时 74.5s
已处理 390/1007 条，耗时 75.4s
已处理 400/1007 条，耗时 76.4s
已处理 410/1007 条，耗时 77.4s
已处理 420/1007 条，耗时 78.3s
已处理 43

### 7.1 评估 LoRA 结果

In [32]:
lora_metric = evaluate_predictions(lora_out["predictions_df"], answer_df)

print(f"Accuracy    : {lora_metric['accuracy']:.4f}")
print(f"Macro-F1    : {lora_metric['macro_f1']:.4f}")
print(f"Weighted-F1 : {lora_metric['weighted_f1']:.4f}")
print()
print(lora_metric["classification_report"])

Accuracy    : 0.6216
Macro-F1    : 0.6214
Weighted-F1 : 0.6223

              precision    recall  f1-score   support

      1 star       0.73      0.66      0.69       212
      2 star       0.53      0.59      0.56       223
      3 star       0.63      0.56      0.60       213
      4 star       0.52      0.53      0.52       169
      5 star       0.71      0.76      0.73       190

    accuracy                           0.62      1007
   macro avg       0.62      0.62      0.62      1007
weighted avg       0.63      0.62      0.62      1007



### 7.2 混淆矩阵

In [33]:
cm_lora = pd.DataFrame(
    lora_metric["confusion_matrix"],
    index=[f"True {i}★" for i in range(1, 6)],
    columns=[f"Pred {i}★" for i in range(1, 6)],
)
print("LoRA 混淆矩阵:")
cm_lora

LoRA 混淆矩阵:


,Pred 1★,Pred 2★,Pred 3★,Pred 4★,Pred 5★
True 1★,140,69,3,0,0
True 2★,49,132,42,0,0
True 3★,2,46,120,43,2
True 4★,1,1,21,90,56
True 5★,1,0,4,41,144


### 7.3 保存预测结果

In [34]:
lora_out["predictions_df"].to_csv(f"{OUTPUT_DIR}/lora_predictions.csv", index=False)
print("已保存 lora_predictions.csv")

已保存 lora_predictions.csv


---
## 8. 结果汇总对比

将三条路线的指标汇总到一张表，方便撰写报告。

In [35]:
comparison = pd.DataFrame([
    {
        "Method"     : "Task A: Zero-shot",
        "Accuracy"   : round(zero_metric["accuracy"],   4),
        "Macro-F1"   : round(zero_metric["macro_f1"],   4),
        "Weighted-F1": round(zero_metric["weighted_f1"],4),
        "Time(s)"    : round(zero_out["inference_seconds"], 1),
        "s/sample"   : round(zero_out["inference_seconds"] / zero_out["sample_count"], 3),
    },
    {
        "Method"     : f"Task B.1.1: {N_SHOT}-shot ICL",
        "Accuracy"   : round(nshot_metric["accuracy"],   4),
        "Macro-F1"   : round(nshot_metric["macro_f1"],   4),
        "Weighted-F1": round(nshot_metric["weighted_f1"],4),
        "Time(s)"    : round(nshot_out["inference_seconds"], 1),
        "s/sample"   : round(nshot_out["inference_seconds"] / nshot_out["sample_count"], 3),
    },
    {
        "Method"     : "Task B.1.2: LoRA Fine-tuned",
        "Accuracy"   : round(lora_metric["accuracy"],   4),
        "Macro-F1"   : round(lora_metric["macro_f1"],   4),
        "Weighted-F1": round(lora_metric["weighted_f1"],4),
        "Time(s)"    : round(lora_out["inference_seconds"], 1),
        "s/sample"   : round(lora_out["inference_seconds"] / lora_out["sample_count"], 3),
    },
])

comparison

,Method,Accuracy,Macro-F1,Weighted-F1,Time(s),s/sample
0,Task A: Zero-shot,0.4816,0.4537,0.4485,118.0,0.117
1,Task B.1.1: 4-shot ICL,0.5065,0.4390,0.4446,135.2,0.134
2,Task B.1.2: LoRA Fine-tuned,0.6216,0.6214,0.6223,172.8,0.172


In [36]:
comparison.to_csv(f"{OUTPUT_DIR}/comparison_results.csv", index=False)
print("已保存 comparison_results.csv")

已保存 comparison_results.csv


---
## 9. Debug：查看原始模型输出

对比不同方法对同一条评论的原始输出，便于分析模型行为。

In [37]:
# 查看详细预测 debug 信息
print("=== Zero-shot debug ===")
for rec in zero_out["debug_records"]:
    print(f"[{rec['review_id']}] pred={rec['predicted_rating']}  raw='{rec['raw_output']}'")

print()
print("=== N-shot debug ===")
for rec in nshot_out["debug_records"]:
    print(f"[{rec['review_id']}] pred={rec['predicted_rating']}  raw='{rec['raw_output']}'")

print()
print("=== LoRA debug ===")
for rec in lora_out["debug_records"]:
    print(f"[{rec['review_id']}] pred={rec['predicted_rating']}  raw='{rec['raw_output']}'")

=== Zero-shot debug ===
[6204917469] pred=1  raw='Rating: 1'
[7226897287] pred=1  raw='Rating: 1'
[1264902203] pred=2  raw='Rating: 2'
[3692481359] pred=2  raw='Rating: 2'
[7677228495] pred=5  raw='Rating: 5'
[1019631414] pred=1  raw='Rating: 1'
[5123417064] pred=2  raw='Rating: 2'
[5676789481] pred=2  raw='Rating: 2'
[9868191978] pred=5  raw='Rating: 5'
[6847353347] pred=1  raw='Rating: 1'
[6568314410] pred=1  raw='Rating: 1'
[5310406185] pred=3  raw='Rating: 3'
[1801252082] pred=1  raw='Rating: 1'
[9739603911] pred=1  raw='Rating: 1'
[3074861604] pred=1  raw='Rating: 1'
[7979374648] pred=4  raw='Rating: 4'
[7951938872] pred=4  raw='Rating: 4'
[6793182662] pred=5  raw='Rating: 5'
[4016545934] pred=5  raw='Rating: 5'
[5543952715] pred=1  raw='Rating: 1'
[1847313872] pred=1  raw='Rating: 1'
[2860929080] pred=2  raw='Rating: 2'
[6249981723] pred=2  raw='Rating: 2'
[8414869219] pred=1  raw='Rating: 1'
[3434596438] pred=1  raw='Rating: 1'
[8842256948] pred=1  raw='Rating: 1'
[4371181493] p